# Tokenizer Comparison Analysis

Analyse runs from the `AJILE12_TOKENIZER_ABLATION` wandb group.

**Research questions:**
- **Q1:** Spatial session vs per-channel?
- **Q2:** CWT vs ResampleCNN vs per-timepoint?
- **Q3:** Is a learned layer after spatial projection useful?
- **Q4:** Is embed_dim a bottleneck for spatial session + CWT?

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

WANDB_PROJECT = "foundry"
WANDB_GROUP = "AJILE12_TOKENIZER_ABLATION"
PRIMARY_METRIC = "val/ajile_active_behavior_auroc"
SECONDARY_METRICS = [
    "val/ajile_active_behavior_f1",
    "val/ajile_active_behavior_balanced_acc",
]
ALL_METRICS = [PRIMARY_METRIC] + SECONDARY_METRICS

## 1. Fetch runs from W&B

In [ ]:
api = wandb.Api()
runs = api.runs(
    WANDB_PROJECT,
    filters={"group": WANDB_GROUP, "state": "finished"},
)
print(f"Found {len(runs)} finished runs in group '{WANDB_GROUP}'")

In [ ]:
records = []
history_frames = []

for run in runs:
    name = run.name  # e.g. ajile_behavior_spatial_session_cwt_fold0
    parts = name.removeprefix("ajile_behavior_").rsplit("_fold", 1)
    tokenizer = parts[0] if len(parts) == 2 else name
    fold = int(parts[1]) if len(parts) == 2 else -1

    summary = run.summary
    record = {
        "run_id": run.id,
        "run_name": name,
        "tokenizer": tokenizer,
        "fold": fold,
    }
    for m in ALL_METRICS:
        record[m] = summary.get(m)
    record["best_epoch"] = summary.get("trainer/current_epoch", summary.get("epoch"))
    records.append(record)

    hist = run.history(keys=[PRIMARY_METRIC, "val/loss", "train/loss", "epoch"], pandas=True)
    hist["tokenizer"] = tokenizer
    hist["fold"] = fold
    history_frames.append(hist)

df = pd.DataFrame(records)
history_df = pd.concat(history_frames, ignore_index=True)
print(f"Parsed {len(df)} runs across {df['tokenizer'].nunique()} tokenizer configs")
df.head(15)

## 2. Annotate configs with metadata

In [ ]:
SPATIAL_MAP = {
    "per_channel_cwt": "PerChannel",
    "per_channel_resample_cnn": "PerChannel",
    "per_channel_per_timepoint_linear": "PerChannel",
    "spatial_session_cwt": "SpatialSession",
    "spatial_session_cwt_common": "SpatialSession",
    "spatial_session_resample_cnn": "SpatialSession",
    "spatial_session_per_timepoint_linear": "SpatialSession",
    "spatial_session_per_timepoint_identity": "SpatialSession",
    "spatial_session_cwt_dim512": "SpatialSession",
    "per_channel_cwt_dim512": "PerChannel",
    "per_channel_per_timepoint_linear_dim512": "PerChannel",
}

TEMPORAL_MAP = {
    "per_channel_cwt": "CWT",
    "per_channel_resample_cnn": "ResampleCNN",
    "per_channel_per_timepoint_linear": "PerTimepoint",
    "spatial_session_cwt": "CWT",
    "spatial_session_cwt_common": "CWT",
    "spatial_session_resample_cnn": "ResampleCNN",
    "spatial_session_per_timepoint_linear": "PerTimepoint",
    "spatial_session_per_timepoint_identity": "PerTimepoint",
    "spatial_session_cwt_dim512": "CWT",
    "per_channel_cwt_dim512": "CWT",
    "per_channel_per_timepoint_linear_dim512": "PerTimepoint",
}

EMBED_DIM_MAP = {
    "spatial_session_cwt_dim512": 512,
    "per_channel_cwt_dim512": 512,
    "per_channel_per_timepoint_linear_dim512": 512,
}

df["spatial"] = df["tokenizer"].map(SPATIAL_MAP)
df["temporal"] = df["tokenizer"].map(TEMPORAL_MAP)
df["embed_dim"] = df["tokenizer"].map(EMBED_DIM_MAP).fillna(256).astype(int)

# Pretty labels for plots
df["label"] = df["tokenizer"].str.replace("_", " ").str.title()

df[["tokenizer", "spatial", "temporal", "embed_dim", "fold", PRIMARY_METRIC]].sort_values(PRIMARY_METRIC, ascending=False)

In [ ]:
agg = (
    df.groupby(["tokenizer", "spatial", "temporal", "embed_dim"])[ALL_METRICS]
    .agg(["mean", "std", "min", "max"])
    .sort_values((PRIMARY_METRIC, "mean"), ascending=False)
)
agg

## 3. Overview: All configs ranked by AUROC

In [ ]:
summary = (
    df.groupby(["tokenizer", "spatial"])[PRIMARY_METRIC]
    .agg(["mean", "min", "max"])
    .reset_index()
    .sort_values("mean", ascending=True)
)
summary["err_lo"] = summary["mean"] - summary["min"]
summary["err_hi"] = summary["max"] - summary["mean"]

palette = {"PerChannel": "#4C72B0", "SpatialSession": "#DD8452"}
colors = [palette[s] for s in summary["spatial"]]

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = range(len(summary))
ax.barh(
    y_pos,
    summary["mean"],
    xerr=[summary["err_lo"], summary["err_hi"]],
    color=colors,
    edgecolor="white",
    capsize=3,
)
ax.set_yticks(y_pos)
ax.set_yticklabels(summary["tokenizer"].str.replace("_", " "), fontsize=9)
ax.set_xlabel(f"Mean {PRIMARY_METRIC}")
ax.set_title("All tokenizer configs ranked by AUROC (mean across folds)")

from matplotlib.patches import Patch
ax.legend(
    handles=[Patch(color=c, label=l) for l, c in palette.items()],
    loc="lower right",
)
fig.tight_layout()
plt.show()

## 4. Q1 -- Spatial Session vs Per-Channel

In [ ]:
q1_configs = [
    "per_channel_cwt", "spatial_session_cwt",
    "per_channel_resample_cnn", "spatial_session_resample_cnn",
    "per_channel_per_timepoint_linear", "spatial_session_per_timepoint_linear",
]
q1 = df[df["tokenizer"].isin(q1_configs) & (df["embed_dim"] == 256)].copy()

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=q1,
    x="temporal",
    y=PRIMARY_METRIC,
    hue="spatial",
    order=["CWT", "ResampleCNN", "PerTimepoint"],
    hue_order=["PerChannel", "SpatialSession"],
    palette=palette,
    capsize=0.05,
    errorbar=("pi", 100),
    ax=ax,
)
ax.set_title("Q1: Spatial Session vs Per-Channel (grouped by temporal embedding)")
ax.set_ylabel(PRIMARY_METRIC)
ax.set_xlabel("Temporal Embedding")
ax.legend(title="Spatial Strategy")
fig.tight_layout()
plt.show()

## 5. Q2 -- CWT vs ResampleCNN vs Per-Timepoint

In [ ]:
q2 = df[df["tokenizer"].isin(q1_configs) & (df["embed_dim"] == 256)].copy()

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=q2,
    x="spatial",
    y=PRIMARY_METRIC,
    hue="temporal",
    order=["PerChannel", "SpatialSession"],
    hue_order=["CWT", "ResampleCNN", "PerTimepoint"],
    capsize=0.05,
    errorbar=("pi", 100),
    ax=ax,
)
ax.set_title("Q2: Temporal embeddings within each spatial strategy")
ax.set_ylabel(PRIMARY_METRIC)
ax.set_xlabel("Spatial Strategy")
ax.legend(title="Temporal Embedding")
fig.tight_layout()
plt.show()

## 6. Q3 -- Learned Post-Projection Layer

In [ ]:
q3a_configs = ["spatial_session_cwt", "spatial_session_cwt_common"]
q3b_configs = ["spatial_session_per_timepoint_identity", "spatial_session_per_timepoint_linear"]

q3a = df[df["tokenizer"].isin(q3a_configs)].copy()
q3a["variant"] = q3a["tokenizer"].map({
    "spatial_session_cwt": "No common layer",
    "spatial_session_cwt_common": "With common layer",
})

q3b = df[df["tokenizer"].isin(q3b_configs)].copy()
q3b["variant"] = q3b["tokenizer"].map({
    "spatial_session_per_timepoint_identity": "Identity (no projection)",
    "spatial_session_per_timepoint_linear": "Linear projection",
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

sns.barplot(
    data=q3a, x="variant", y=PRIMARY_METRIC,
    capsize=0.05, errorbar=("pi", 100), ax=axes[0],
)
axes[0].set_title("Q3a: Common layer ablation (CWT)")
axes[0].set_xlabel("")
axes[0].set_ylabel(PRIMARY_METRIC)

# Annotate delta
means_a = q3a.groupby("variant")[PRIMARY_METRIC].mean()
if len(means_a) == 2:
    delta_a = means_a.iloc[1] - means_a.iloc[0]
    axes[0].annotate(
        f"\u0394 = {delta_a:+.4f}",
        xy=(0.5, 0.95), xycoords="axes fraction",
        ha="center", fontsize=11, fontweight="bold",
    )

sns.barplot(
    data=q3b, x="variant", y=PRIMARY_METRIC,
    capsize=0.05, errorbar=("pi", 100), ax=axes[1],
)
axes[1].set_title("Q3b: Temporal projection ablation (PerTimepoint)")
axes[1].set_xlabel("")
axes[1].set_ylabel("")

means_b = q3b.groupby("variant")[PRIMARY_METRIC].mean()
if len(means_b) == 2:
    delta_b = means_b.iloc[1] - means_b.iloc[0]
    axes[1].annotate(
        f"\u0394 = {delta_b:+.4f}",
        xy=(0.5, 0.95), xycoords="axes fraction",
        ha="center", fontsize=11, fontweight="bold",
    )

fig.suptitle("Q3: Is a learned layer after spatial projection useful?", fontsize=13)
fig.tight_layout()
plt.show()

## 7. Q4 -- Embed Dim Scaling

In [ ]:
q4_groups = {
    "Spatial Session CWT": ["spatial_session_cwt", "spatial_session_cwt_dim512"],
    "Per-Channel CWT": ["per_channel_cwt", "per_channel_cwt_dim512"],
    "Per-Channel PerTimepoint": ["per_channel_per_timepoint_linear", "per_channel_per_timepoint_linear_dim512"],
}

q4 = df[df["tokenizer"].isin(
    [t for group in q4_groups.values() for t in group]
)].copy()

q4["config_group"] = q4["tokenizer"].map(
    {t: grp for grp, ts in q4_groups.items() for t in ts}
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=q4,
    x="config_group",
    y=PRIMARY_METRIC,
    hue="embed_dim",
    order=["Spatial Session CWT", "Per-Channel CWT", "Per-Channel PerTimepoint"],
    capsize=0.05,
    errorbar=("pi", 100),
    ax=ax,
)
ax.set_title("Q4: Does larger embed_dim help? (spatial session vs per-channel controls)")
ax.set_ylabel(PRIMARY_METRIC)
ax.set_xlabel("")
ax.legend(title="embed_dim")
fig.tight_layout()
plt.show()

## 8. Training Curves

In [ ]:
curve_groups = {
    "Q1+Q2: Core factorial (embed_dim=256)": q1_configs,
    "Q3a: Common layer ablation": q3a_configs,
    "Q3b: Temporal projection ablation": q3b_configs,
    "Q4: Embed dim scaling": [
        "spatial_session_cwt", "spatial_session_cwt_dim512",
        "per_channel_cwt", "per_channel_cwt_dim512",
        "per_channel_per_timepoint_linear", "per_channel_per_timepoint_linear_dim512",
    ],
}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for ax, (title, configs) in zip(axes.flat, curve_groups.items()):
    subset = history_df[history_df["tokenizer"].isin(configs)].copy()
    if subset.empty:
        ax.set_title(f"{title} (no data)")
        continue
    subset["run_label"] = subset["tokenizer"] + " fold" + subset["fold"].astype(str)
    for label, grp in subset.groupby("run_label"):
        grp_sorted = grp.dropna(subset=[PRIMARY_METRIC]).sort_values("_step")
        if grp_sorted.empty:
            continue
        ax.plot(grp_sorted["_step"], grp_sorted[PRIMARY_METRIC], label=label, alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Step")
    ax.set_ylabel(PRIMARY_METRIC)
    ax.legend(fontsize=7, ncol=2, loc="lower right")

fig.suptitle("Training curves: val AUROC over training", fontsize=14)
fig.tight_layout()
plt.show()

## 9. Summary Table

In [ ]:
summary_table = (
    df.groupby(["tokenizer", "spatial", "temporal", "embed_dim"])
    .agg(
        auroc_mean=(PRIMARY_METRIC, "mean"),
        auroc_std=(PRIMARY_METRIC, "std"),
        f1_mean=("val/ajile_active_behavior_f1", "mean"),
        f1_std=("val/ajile_active_behavior_f1", "std"),
        balanced_acc_mean=("val/ajile_active_behavior_balanced_acc", "mean"),
        balanced_acc_std=("val/ajile_active_behavior_balanced_acc", "std"),
        best_epoch_mean=("best_epoch", "mean"),
    )
    .sort_values("auroc_mean", ascending=False)
    .reset_index()
)

def highlight_best(s):
    if s.name.endswith("_mean"):
        is_best = s == s.max()
        return ["font-weight: bold" if v else "" for v in is_best]
    return ["" for _ in s]

summary_table.style.apply(highlight_best).format(precision=4)